<a href="https://colab.research.google.com/github/KaKigw/Transcoder_Interp/blob/main/04_gpt2_transcoder_pilot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

| Step | What we’ll do | Why it matters |
|---|---|---|
| **1. Fix the experiment** | Choose one GPT-2 MLP layer, dataset, feature widths, sparsity settings, and training budget. Align with your teammate’s setup where practical. | Makes the experiment reproducible and comparable. |
| **2. Cache MLP inputs and outputs** | Collect pairs \(x \rightarrow \mathrm{MLP}(x)\), with separate training, validation, and test documents. | Prevents data leakage and avoids repeatedly running GPT-2 during training. |
| **3. Validate the evaluator** | Check identity interventions, direct MLP evaluation, tensor shapes, and response calculations. | Ensures apparent differences aren’t evaluation bugs. |
| **4. Train the three variants** | Learned, frozen encoder, and frozen decoder, with consistent initialization and data order. Record exactly which biases train and how decoder norms are handled. | Defines what our comparison actually tests. |
| **5. Check optimization** | For learned and frozen-encoder models, freeze their feature maps and refit unrestricted decoders using training data. Keep these as separate diagnostic branches. | Carries forward the lesson from our toy experiment. |
| **6. Build reconstruction–sparsity curves** | Measure validation reconstruction error and actual L0 across sparsity settings. Identify comparable models using validation data only. | Avoids attributing better reconstruction or more active features to better directions. |
| **7. Evaluate intervention responses** | Apply identical input edits to the original MLP and each transcoder. Compare their output changes on a fixed panel. | Tests the behavior we care about. |
| **8. Confirm and decide** | Repeat promising comparisons across seeds, lock model selection, then evaluate on held-out test data with uncertainty estimates. | Determines whether the result is reliable enough to continue. |

In [ ]:
from pathlib import Path
import subprocess
import sys
import torch
repo_path = Path("/content/transcoder_circuits")

if not repo_path.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/jacobdunefsky/transcoder_circuits.git",
            str(repo_path),
        ],
        check=True,
    )

commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=repo_path,
    text=True,
).strip()

print("Repository:", repo_path)
print("Commit:", commit)
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

print("\nRepository requirements:")
print((repo_path / "requirements.txt").read_text())

Repository: /content/transcoder_circuits
Commit: 8721ff06b64889e8cc50bb1c5e869c7f72b8303d
Python: 3.13.15
PyTorch: 2.11.0+cu130
GPU available: True

Repository requirements:
matplotlib
numpy
plotly
torch==2.2.0
tqdm
transformer_lens==1.11.0
datasets==2.12.0
einops==0.7.0
setuptools==67.7.2
wandb==0.16.0
huggingface-hub==0.17.3


In [ ]:
epo_path = Path("/content/transcoder_circuits")
# Preserve the currently installed PyTorch version.
constraints_path = repo_path / "pilot_constraints.txt"
constraints_path.write_text(f"torch=={torch.__version__}\n")

subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--constraint", str(constraints_path),
        "transformer-lens==3.2.1",
        "einops",
        "jaxtyping",
        "wandb",
    ],
    check=True,
)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '--constraint', '/content/transcoder_circuits/pilot_constraints.txt', 'transformer-lens==3.2.1', 'einops', 'jaxtyping', 'wandb'], returncode=0)

In [ ]:
from sae_training.config import RunnerConfig

In [ ]:
from pathlib import Path
import sys
import torch

repo_path = Path("/content/transcoder_circuits")
assert repo_path.exists(), f"Repository not found: {repo_path}"

if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))

from transformer_lens import HookedTransformer
from sae_training.config import LanguageModelSAERunnerConfig
from sae_training.sparse_autoencoder import SparseAutoencoder

print("Repository imports passed.")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

assert torch.cuda.is_available(), (
    "Enable a GPU: Runtime → Change runtime type → GPU."
)

device = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(device))

Repository imports passed.
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [ ]:
import random
import numpy as np
import torch
from transformer_lens import HookedTransformer

SEED = 42
LAYER = 5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Keep the original layer-normalization parameterization.
lm = HookedTransformer.from_pretrained(
    "gpt2-small",
    device=device,
    dtype=torch.float32,
    fold_ln=False,
    center_writing_weights=False,
    center_unembed=False,
    refactor_factored_attn_matrices=False,
)

lm.eval()
lm.requires_grad_(False)

print("Model:", lm.cfg.model_name)
print("Transformer blocks:", lm.cfg.n_layers)
print("Activation dimension:", lm.cfg.d_model)
print("MLP hidden dimension:", lm.cfg.d_mlp)
print("Selected block:", LAYER)
print("Trainable parameters:", sum(
    p.numel() for p in lm.parameters() if p.requires_grad
))

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Loaded pretrained model gpt2-small into HookedTransformer
Model: gpt2
Transformer blocks: 12
Activation dimension: 768
MLP hidden dimension: 3072
Selected block: 5
Trainable parameters: 0


In [ ]:
captured = {}
target_mlp = lm.blocks[LAYER].mlp

def capture_input(module, args):
    captured["input"] = args[0].detach().clone()

def capture_output(module, args, output):
    captured["output"] = output.detach().clone()

input_handle = target_mlp.register_forward_pre_hook(capture_input)
output_handle = target_mlp.register_forward_hook(capture_output)

try:
    tokens = lm.to_tokens(
        "The scientist carefully examined the results of the experiment.",
        prepend_bos=True,
    )

    with torch.no_grad():
        lm(tokens, return_type=None)
finally:
    # Remove hooks so later calls don't overwrite these captures.
    input_handle.remove()
    output_handle.remove()

x_check = captured["input"]
y_check = captured["output"]

# Feeding the captured input directly into the MLP should reproduce
# the output captured during the full GPT-2 forward pass.
with torch.no_grad():
    y_direct = target_mlp(x_check)

max_error = (y_direct - y_check).abs().max().item()

print("Token shape:", tuple(tokens.shape))
print("MLP input shape:", tuple(x_check.shape))
print("MLP output shape:", tuple(y_check.shape))
print(f"Direct MLP reproduction — max error: {max_error:.6e}")

assert x_check.shape == y_check.shape
assert x_check.shape[-1] == 768
assert torch.isfinite(x_check).all()
assert torch.isfinite(y_check).all()
torch.testing.assert_close(y_direct, y_check, rtol=1e-5, atol=1e-6)

print("MLP extraction check passed.")

Token shape: (1, 11)
MLP input shape: (1, 11, 768)
MLP output shape: (1, 11, 768)
Direct MLP reproduction — max error: 0.000000e+00
MLP extraction check passed.


In [ ]:
from datasets import load_dataset

DATASET_NAME = "Salesforce/wikitext"
DATASET_CONFIG = "wikitext-103-raw-v1"

raw_data = load_dataset(
    DATASET_NAME,
    DATASET_CONFIG,
    split={
        "train": "train",
        "validation": "validation",
    },
)

for split_name, dataset in raw_data.items():
    nonempty = sum(bool(text.strip()) for text in dataset["text"])
    print(
        f"{split_name:10s} | Rows: {len(dataset):,}"
        f" | Nonempty rows: {nonempty:,}"
    )

print("\nExample training text:")
for row in raw_data["train"]:
    text = row["text"].strip()
    if len(text) > 200:
        print(text[:500])
        break

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-103-raw-v1/test-00000-of-00001.(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-103-raw-v1/test-00000-of-00001.(…): downloading bytes:           |  0.00B            

wikitext-103-raw-v1/train-00000-of-00002(…): reconstructing file:   0%|          |  0.00B /  157MB            

wikitext-103-raw-v1/train-00000-of-00002(…): downloading bytes:           |  0.00B            

wikitext-103-raw-v1/train-00001-of-00002(…): reconstructing file:   0%|          |  0.00B /  157MB            

... (3 outputs left out)

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1801350 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

train      | Rows: 1,801,350 | Nonempty rows: 1,165,029
validation | Rows: 3,760 | Nonempty rows: 2,461

Example training text:
Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and fo


In [ ]:
import re
import hashlib

# Match single-level titles, excluding subsection headings.
article_title_pattern = re.compile(
    r"^=\s*([^=].*?)\s*=$"
)

def is_article_title(text):
    text = text.strip()
    return (
        bool(article_title_pattern.fullmatch(text))
        and not text.lstrip("= ").startswith("=")
        and text.count("=") == 2
    )


def group_articles(dataset, split_name):
    articles = []
    current_title = None
    current_lines = []

    def save_article():
        if current_title is None:
            return

        body = "\n".join(current_lines).strip()
        if not body:
            return

        text = current_title + "\n" + body
        articles.append({
            "article_id": f"{split_name}:{len(articles)}",
            "title": current_title,
            "text": text,
            "text_hash": hashlib.sha256(text.encode("utf-8")).hexdigest(),
        })

    for row in dataset:
        text = row["text"].strip()
        if not text:
            continue

        if is_article_title(text):
            save_article()
            current_title = text
            current_lines = []
        elif current_title is not None:
            current_lines.append(text)
        else:
            raise ValueError("Found text before the first article title.")

    save_article()
    return articles


articles = {
    split: group_articles(dataset, split)
    for split, dataset in raw_data.items()
}

for split, items in articles.items():
    print(f"{split}: {len(items):,} articles")
    print("  First title:", items[0]["title"])

# Check for exact duplicate articles across training and validation.
train_hashes = {a["text_hash"] for a in articles["train"]}
val_hashes = {a["text_hash"] for a in articles["validation"]}
overlap = train_hashes & val_hashes

print("\nExact article overlap:", len(overlap))
assert not overlap, "Duplicate articles found across splits."


train: 29,443 articles
  First title: = Valkyria Chronicles III =
validation: 60 articles
  First title: = Homarus gammarus =

Exact article overlap: 0


In [ ]:
CONTEXT_SIZE = 128
TEXT_TOKENS_PER_WINDOW = CONTEXT_SIZE - 1

def prepare_windows(article_list, n_articles, n_windows, seed):
    rng = random.Random(seed)

    # Select articles reproducibly.
    selected = rng.sample(
        article_list, min(n_articles, len(article_list))
    )

    pools = []
    for article in selected:
        token_ids = lm.tokenizer.encode(
            article["text"],
            add_special_tokens=False,
            truncation=False,
            verbose=False,
        )

        # Non-overlapping, full-length chunks within this article.
        starts = list(range(
            0,
            len(token_ids) - TEXT_TOKENS_PER_WINDOW + 1,
            TEXT_TOKENS_PER_WINDOW,
        ))
        rng.shuffle(starts)

        if starts:
            pools.append({
                "article_id": article["article_id"],
                "token_ids": token_ids,
                "starts": starts,
            })

    windows, article_ids, offsets = [], [], []

    # Take one window per article per round.
    while len(windows) < n_windows:
        made_progress = False

        for pool in pools:
            if not pool["starts"]:
                continue

            start = pool["starts"].pop()
            chunk = pool["token_ids"][
                start : start + TEXT_TOKENS_PER_WINDOW
            ]

            windows.append([lm.tokenizer.bos_token_id] + chunk)
            article_ids.append(pool["article_id"])
            offsets.append(start)
            made_progress = True

            if len(windows) == n_windows:
                break

        if not made_progress:
            raise ValueError("Not enough full windows in selected articles.")

    return {
        "tokens": torch.tensor(windows, dtype=torch.long),
        "article_ids": article_ids,
        "article_token_offsets": torch.tensor(offsets),
        "seed": seed,
    }


window_data = {
    "train": prepare_windows(
        articles["train"], n_articles=256,
        n_windows=1024, seed=1001,
    ),
    "validation": prepare_windows(
        articles["validation"], n_articles=60,
        n_windows=256, seed=1002,
    ),
}

for split, data in window_data.items():
    tokens = data["tokens"]

    assert tokens.shape[1] == CONTEXT_SIZE
    assert (tokens[:, 0] == lm.tokenizer.bos_token_id).all()

    print(
        f"{split:10s} | Windows: {len(tokens):,}"
        f" | Articles: {len(set(data['article_ids']))}"
        f" | Usable token positions: {len(tokens) * 127:,}"
    )

print("\nExample training window:")
print(lm.tokenizer.decode(window_data["train"]["tokens"][0].tolist()))

train      | Windows: 1,024 | Articles: 251 | Usable token positions: 130,048
validation | Windows: 256 | Articles: 60 | Usable token positions: 32,512

Example training window:
<|endoftext|> that was never aired plus a visit to WWE Raw . During the publicity run , Dennis Phillips served as Cada 's advisor .
He has a total of seven cashes in the World Series of Poker , with three in the 2009 event , two in the 2011 event and two in the 2012 event . Aside from his main event win , his most notable WSOP finish was second in the 2811 @-@ entrant June 16 – 18 , 2012 $ 1 @,@ 500 No @-@ Limit Hold 'em Event # 31 , which earned him $ 412 @,@ 424 .
= = Personal = =
Anne


In [ ]:
from google.colab import drive
from pathlib import Path
from tqdm.auto import tqdm

drive.mount("/content/drive")

cache_dir = Path(
    "/content/drive/MyDrive/transcoder_pilot/gpt2_block5/cache"
)
cache_dir.mkdir(parents=True, exist_ok=True)


def cache_mlp_pairs(window_batch, batch_size=8):
    tokens = window_batch["tokens"]
    n_positions = len(tokens) * TEXT_TOKENS_PER_WINDOW

    # Allocate on CPU so the entire cache doesn't occupy GPU memory.
    inputs = torch.empty(n_positions, lm.cfg.d_model, dtype=torch.float32)
    outputs = torch.empty_like(inputs)
    captured_batch = {}

    def capture_in(module, args):
        captured_batch["x"] = args[0].detach()

    def capture_out(module, args, output):
        captured_batch["y"] = output.detach()

    h_in = target_mlp.register_forward_pre_hook(capture_in)
    h_out = target_mlp.register_forward_hook(capture_out)

    cursor = 0
    try:
        with torch.no_grad():
            for start in tqdm(range(0, len(tokens), batch_size)):
                batch = tokens[start:start + batch_size].to(device)

                # Stop after block 5: later blocks aren't needed.
                lm(batch, return_type=None, stop_at_layer=LAYER + 1)

                # Exclude BOS, retaining the same ordering for x and y.
                x = captured_batch["x"][:, 1:, :].reshape(-1, lm.cfg.d_model)
                y = captured_batch["y"][:, 1:, :].reshape(-1, lm.cfg.d_model)

                count = len(x)
                inputs[cursor:cursor + count].copy_(x.cpu())
                outputs[cursor:cursor + count].copy_(y.cpu())
                cursor += count
    finally:
        h_in.remove()
        h_out.remove()

    assert cursor == n_positions
    assert torch.isfinite(inputs).all()
    assert torch.isfinite(outputs).all()

    return {
        "inputs": inputs,
        "outputs": outputs,
        # Row i maps to window i // 127 and position 1 + i % 127.
        "windows": window_batch,
        "metadata": {
            "model": "gpt2-small",
            "layer": LAYER,
            "context_size": CONTEXT_SIZE,
            "exclude_bos": True,
            "fold_ln": False,
            "center_writing_weights": False,
            "center_unembed": False,
            "refactor_factored_attn_matrices": False,
            "dataset": DATASET_NAME,
            "dataset_config": DATASET_CONFIG,
        },
    }


activation_cache = {}

for split, windows in window_data.items():
    path = cache_dir / f"{split}_initial_float32.pt"

    # Avoid silently overwriting an existing experiment.
    if path.exists():
        raise FileExistsError(f"Cache already exists: {path}")

    print(f"\nCaching {split}...")
    cached = cache_mlp_pairs(windows)
    torch.save(cached, path)
    activation_cache[split] = cached

    print("Input shape:", tuple(cached["inputs"].shape))
    print("Output shape:", tuple(cached["outputs"].shape))
    print("Saved:", path)

Mounted at /content/drive

Caching train...


  0%|          | 0/128 [00:00<?, ?it/s]

Input shape: (130048, 768)
Output shape: (130048, 768)
Saved: /content/drive/MyDrive/transcoder_pilot/gpt2_block5/cache/train_initial_float32.pt

Caching validation...


  0%|          | 0/32 [00:00<?, ?it/s]

Input shape: (32512, 768)
Output shape: (32512, 768)
Saved: /content/drive/MyDrive/transcoder_pilot/gpt2_block5/cache/validation_initial_float32.pt


In [ ]:
cache_check_rng = torch.Generator().manual_seed(2001)

for split in ["train", "validation"]:
    path = cache_dir / f"{split}_initial_float32.pt"
    saved = torch.load(path, map_location="cpu", weights_only=True)

    x = saved["inputs"]
    y = saved["outputs"]

    assert x.shape == y.shape
    assert x.shape[1] == 768

    indices = torch.randperm(len(x), generator=cache_check_rng)[:64]

    with torch.no_grad():
        # Treat each sampled activation as a one-token MLP input.
        replayed = target_mlp(
            x[indices].to(device).unsqueeze(1)
        ).squeeze(1).cpu()

    reference = y[indices]
    difference = replayed - reference

    relative_error = (
        difference.norm() / reference.norm().clamp_min(1e-12)
    ).item()

    print(
        f"{split:10s} | Pairs: {len(x):,}"
        f" | Max absolute error: {difference.abs().max().item():.3e}"
        f" | Relative error: {relative_error:.3e}"
    )

    # Different GPU batch shapes can produce small rounding differences.
    assert relative_error < 1e-5, "Cached-pair reproduction failed."

    del saved, x, y

print("\nSaved cache verification passed.")


train      | Pairs: 130,048 | Max absolute error: 1.192e-05 | Relative error: 1.225e-06
validation | Pairs: 32,512 | Max absolute error: 1.216e-05 | Relative error: 1.179e-06

Saved cache verification passed.


In [ ]:
import json

pilot_config = {
    "model": "gpt2-small",
    "layer": LAYER,
    "d_in": 768,
    "d_out": 768,
    "n_features": 3072,
    "activation": "TopK_after_ReLU",
    "k_values": [32, 64, 128],
    "variants": ["Learned", "Frozen encoder", "Frozen decoder"],
    "initial_seed": 0,
    "confirmation_seeds": [1, 2],
    "batch_size": 512,
    "training_steps": 2000,
    "learning_rate": 3e-4,
    "biases_trainable": True,
    "decoder_unit_initialization": True,
    "decoder_normalize_after_step": False,
    "l1_penalty": 0.0,
    "primary_intervention": "different_article_donor_blend",
    "primary_blend_alpha": 0.5,
    "meaningful_response_advantage": 0.05,
    "relative_nmse_matching_tolerance": 0.05,
    "match_same_k": True,
    "max_initial_runs": 9,
    "max_confirmation_runs": 6,
    "test_used_for_selection": False,
}

config_path = cache_dir.parent / "pilot_config.json"

if config_path.exists():
    existing = json.loads(config_path.read_text())
    assert existing == pilot_config, (
        "An existing configuration differs. Inspect it before changing it."
    )
else:
    config_path.write_text(json.dumps(pilot_config, indent=2))

print(json.dumps(pilot_config, indent=2))
print("\nConfiguration saved:", config_path)

{
  "model": "gpt2-small",
  "layer": 5,
  "d_in": 768,
  "d_out": 768,
  "n_features": 3072,
  "activation": "TopK_after_ReLU",
  "k_values": [
    32,
    64,
    128
  ],
  "variants": [
    "Learned",
    "Frozen encoder",
    "Frozen decoder"
  ],
  "initial_seed": 0,
  "confirmation_seeds": [
    1,
    2
  ],
  "batch_size": 512,
  "training_steps": 2000,
  "learning_rate": 0.0003,
  "biases_trainable": true,
  "decoder_unit_initialization": true,
  "decoder_normalize_after_step": false,
  "l1_penalty": 0.0,
  "primary_intervention": "different_article_donor_blend",
  "primary_blend_alpha": 0.5,
  "meaningful_response_advantage": 0.05,
  "relative_nmse_matching_tolerance": 0.05,
  "match_same_k": true,
  "max_initial_runs": 9,
  "max_confirmation_runs": 6,
  "test_used_for_selection": false
}

Configuration saved: /content/drive/MyDrive/transcoder_pilot/gpt2_block5/pilot_config.json


In [ ]:
import copy
import torch.nn as nn
import torch.nn.functional as F


class TopKTranscoder(nn.Module):
    def __init__(self, d_in, n_features, d_out, k, seed):
        super().__init__()
        self.k = k

        generator = torch.Generator().manual_seed(seed)

        # Each encoder column is one input feature direction.
        W_enc = torch.randn(d_in, n_features, generator=generator)
        W_enc = F.normalize(W_enc, dim=0)

        # Each decoder row is one output feature direction.
        W_dec = torch.randn(n_features, d_out, generator=generator)
        W_dec = F.normalize(W_dec, dim=1)

        self.W_enc = nn.Parameter(W_enc)
        self.W_dec = nn.Parameter(W_dec)

        self.b_dec = nn.Parameter(torch.zeros(d_in))
        self.b_enc = nn.Parameter(torch.zeros(n_features))
        self.b_dec_out = nn.Parameter(torch.zeros(d_out))

    def forward(self, x):
        scores = (x - self.b_dec) @ self.W_enc + self.b_enc
        positive = F.relu(scores)

        values, indices = positive.topk(self.k, dim=-1)
        features = torch.zeros_like(positive).scatter(
            dim=-1, index=indices, src=values
        )

        predictions = features @ self.W_dec + self.b_dec_out
        return predictions, features


# Start with k=32. These models are not trained yet.
base_model = TopKTranscoder(
    d_in=pilot_config["d_in"],
    n_features=pilot_config["n_features"],
    d_out=pilot_config["d_out"],
    k=pilot_config["k_values"][0],
    seed=pilot_config["initial_seed"],
)

pilot_models = {
    name: copy.deepcopy(base_model).to(device)
    for name in pilot_config["variants"]
}

pilot_models["Frozen encoder"].W_enc.requires_grad_(False)
pilot_models["Frozen decoder"].W_dec.requires_grad_(False)

# Preserve copies for later exact frozen-matrix checks.
frozen_matrices = {
    "Frozen encoder": pilot_models["Frozen encoder"].W_enc.detach().clone(),
    "Frozen decoder": pilot_models["Frozen decoder"].W_dec.detach().clone(),
}

x_sample = activation_cache["train"]["inputs"][:16].to(device)
reference_prediction = None

with torch.no_grad():
    for name, model in pilot_models.items():
        prediction, features = model(x_sample)
        l0 = (features > 0).sum(dim=-1)

        assert prediction.shape == (16, 768)
        assert features.shape == (16, 3072)
        assert (l0 <= model.k).all()
        assert torch.isfinite(prediction).all()

        if reference_prediction is None:
            reference_prediction = prediction.clone()
        else:
            torch.testing.assert_close(
                prediction, reference_prediction, rtol=0, atol=0
            )

        print(
            f"{name:15s} | Output: {tuple(prediction.shape)}"
            f" | Mean L0: {l0.float().mean().item():.2f}"
            f" | Encoder trains: {model.W_enc.requires_grad}"
            f" | Decoder trains: {model.W_dec.requires_grad}"
        )

print("\nAll variants start with identical predictions.")
print("No training performed.")

Learned         | Output: (16, 768) | Mean L0: 32.00 | Encoder trains: True | Decoder trains: True
Frozen encoder  | Output: (16, 768) | Mean L0: 32.00 | Encoder trains: False | Decoder trains: True
Frozen decoder  | Output: (16, 768) | Mean L0: 32.00 | Encoder trains: True | Decoder trains: False

All variants start with identical predictions.
No training performed.


In [ ]:
# 1. Check gradients without taking an optimizer step.
y_sample = activation_cache["train"]["outputs"][:16].to(device)

for name, model in pilot_models.items():
    model.zero_grad(set_to_none=True)

    prediction, _ = model(x_sample)
    loss = F.mse_loss(prediction, y_sample)
    loss.backward()

    for parameter_name, parameter in model.named_parameters():
        if parameter.requires_grad:
            assert parameter.grad is not None, (
                f"{name}: missing gradient for {parameter_name}"
            )
            assert torch.isfinite(parameter.grad).all()
        else:
            assert parameter.grad is None, (
                f"{name}: frozen parameter received a gradient"
            )

    print(f"{name}: gradient checks passed.")
    model.zero_grad(set_to_none=True)


# 2. Compute validation reference variance in batches.
val_cache = activation_cache["validation"]
val_mean = val_cache["outputs"].mean(dim=0, dtype=torch.float64)

val_reference_sse = 0.0
for target_batch in val_cache["outputs"].split(1024):
    val_reference_sse += (
        target_batch.double() - val_mean
    ).square().sum().item()

assert val_reference_sse > 0


@torch.no_grad()
def validation_metrics(model, batch_size=512):
    was_training = model.training
    model.eval()

    squared_error = 0.0
    active_count = 0
    feature_seen = torch.zeros(
        model.W_enc.shape[1], dtype=torch.bool, device=device
    )

    try:
        for start in range(0, len(val_cache["inputs"]), batch_size):
            x = val_cache["inputs"][start:start + batch_size].to(device)
            y = val_cache["outputs"][start:start + batch_size].to(device)

            prediction, features = model(x)
            squared_error += (
                prediction.double() - y.double()
            ).square().sum().item()

            active = features > 0
            active_count += active.sum().item()
            feature_seen |= active.any(dim=0)
    finally:
        model.train(was_training)

    n_examples = len(val_cache["inputs"])

    return {
        "mse": squared_error / (n_examples * val_cache["outputs"].shape[1]),
        "nmse": squared_error / val_reference_sse,
        "l0": active_count / n_examples,
        "inactive_fraction": (~feature_seen).float().mean().item(),
    }


initial_metrics = {}

print("\nBefore training:")
for name, model in pilot_models.items():
    metrics = validation_metrics(model)
    initial_metrics[name] = metrics

    print(
        f"{name:15s} | Val NMSE: {metrics['nmse']:.4f}"
        f" | L0: {metrics['l0']:.2f}"
        f" | Never active on validation: {metrics['inactive_fraction']:.1%}"
    )

print("\nNo parameter updates performed.")

Learned: gradient checks passed.
Frozen encoder: gradient checks passed.
Frozen decoder: gradient checks passed.

Before training:
Learned         | Val NMSE: 1.0756 | L0: 32.00 | Never active on validation: 3.6%
Frozen encoder  | Val NMSE: 1.0756 | L0: 32.00 | Never active on validation: 3.6%
Frozen decoder  | Val NMSE: 1.0756 | L0: 32.00 | Never active on validation: 3.6%

No parameter updates performed.


In [ ]:
run_dir = cache_dir.parent / "runs" / "k32_seed0"
run_dir.mkdir(parents=True, exist_ok=True)
latest_path = run_dir / "latest.pt"

# Initialize once. Rerunning this cell won't reset the optimizers.
if "pilot_optimizers" not in globals():
    if latest_path.exists():
        raise RuntimeError(
            "A checkpoint already exists. Restore it before continuing."
        )

    pilot_optimizers = {
        name: torch.optim.Adam(
            [p for p in model.parameters() if p.requires_grad],
            lr=pilot_config["learning_rate"],
        )
        for name, model in pilot_models.items()
    }

    batch_generator = torch.Generator().manual_seed(3001)
    completed_steps = 0
    training_history = []
    best_validation = {
        name: initial_metrics[name]["nmse"] for name in pilot_models
    }
    best_states = {
        name: {
            key: value.detach().cpu().clone()
            for key, value in model.state_dict().items()
        }
        for name, model in pilot_models.items()
    }
    best_steps = {name: 0 for name in pilot_models}


def save_training_checkpoint():
    payload = {
        "config": pilot_config,
        "k": 32,
        "seed": 0,
        "completed_steps": completed_steps,
        "models": {
            name: {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            for name, model in pilot_models.items()
        },
        "optimizers": {
            name: optimizer.state_dict()
            for name, optimizer in pilot_optimizers.items()
        },
        "requires_grad": {
            name: {key: p.requires_grad for key, p in model.named_parameters()}
            for name, model in pilot_models.items()
        },
        "batch_generator_state": batch_generator.get_state(),
        "frozen_matrices": {
            name: tensor.cpu().clone()
            for name, tensor in frozen_matrices.items()
        },
        "history": training_history,
        "best_validation": best_validation,
        "best_states": best_states,
        "best_steps": best_steps,
        "sampling": "uniform token sampling with replacement",
    }

    temporary_path = run_dir / "latest.tmp.pt"
    torch.save(payload, temporary_path)
    temporary_path.replace(latest_path)


# Save the initial state before training begins.
if completed_steps == 0:
    save_training_checkpoint()

train_cache = activation_cache["train"]
TARGET_STEP = 2000 # Absolute step number, not 200 additional steps.

for step in tqdm(range(completed_steps + 1, TARGET_STEP + 1)):
    # All three variants receive this same batch.
    indices = torch.randint(
        len(train_cache["inputs"]),
        (pilot_config["batch_size"],),
        generator=batch_generator,
    )
    x = train_cache["inputs"][indices].to(device)
    y = train_cache["outputs"][indices].to(device)

    for name, model in pilot_models.items():
        model.train()
        optimizer = pilot_optimizers[name]
        optimizer.zero_grad(set_to_none=True)

        prediction, _ = model(x)
        loss = F.mse_loss(prediction, y)
        assert torch.isfinite(loss), f"Nonfinite loss: {name}"

        loss.backward()
        optimizer.step()

    completed_steps = step

    if step % 100 == 0:
        assert torch.equal(
            pilot_models["Frozen encoder"].W_enc,
            frozen_matrices["Frozen encoder"],
        )
        assert torch.equal(
            pilot_models["Frozen decoder"].W_dec,
            frozen_matrices["Frozen decoder"],
        )

        print(f"\nStep {step}")
        for name, model in pilot_models.items():
            metrics = validation_metrics(model)
            training_history.append({
                "step": step, "variant": name, **metrics
            })

            if metrics["nmse"] < best_validation[name]:
                best_validation[name] = metrics["nmse"]
                best_steps[name] = step
                best_states[name] = {
                    key: value.detach().cpu().clone()
                    for key, value in model.state_dict().items()
                }

            print(
                f"{name:15s} | Val NMSE: {metrics['nmse']:.4f}"
                f" | L0: {metrics['l0']:.2f}"
                f" | Inactive on val: {metrics['inactive_fraction']:.1%}"
            )

        save_training_checkpoint()
        print("Checkpoint saved; frozen matrices unchanged.")


  0%|          | 0/1800 [00:00<?, ?it/s]


Step 300
Learned         | Val NMSE: 0.8347 | L0: 32.00 | Inactive on val: 12.9%
Frozen encoder  | Val NMSE: 0.9270 | L0: 32.00 | Inactive on val: 4.2%
Frozen decoder  | Val NMSE: 0.9933 | L0: 32.00 | Inactive on val: 3.9%
Checkpoint saved; frozen matrices unchanged.

Step 400
Learned         | Val NMSE: 0.8057 | L0: 32.00 | Inactive on val: 17.9%
Frozen encoder  | Val NMSE: 0.9110 | L0: 32.00 | Inactive on val: 6.7%
Frozen decoder  | Val NMSE: 0.9878 | L0: 32.00 | Inactive on val: 6.5%
Checkpoint saved; frozen matrices unchanged.

Step 500
Learned         | Val NMSE: 0.7821 | L0: 32.00 | Inactive on val: 20.0%
Frozen encoder  | Val NMSE: 0.8999 | L0: 32.00 | Inactive on val: 9.3%
Frozen decoder  | Val NMSE: 0.9837 | L0: 32.00 | Inactive on val: 8.6%
Checkpoint saved; frozen matrices unchanged.

Step 600
Learned         | Val NMSE: 0.7624 | L0: 32.00 | Inactive on val: 20.1%
Frozen encoder  | Val NMSE: 0.8911 | L0: 32.00 | Inactive on val: 11.6%
Frozen decoder  | Val NMSE: 0.9803 | L0

In [ ]:
from collections import defaultdict

panel_path = cache_dir.parent / "validation_intervention_panel.pt"

if panel_path.exists():
    intervention_panel = torch.load(
        panel_path, map_location="cpu", weights_only=True
    )
    print("Loaded existing intervention panel.")
else:
    rng = random.Random(4001)
    windows = val_cache["windows"]
    windows_by_article = defaultdict(list)

    for window_index, article_id in enumerate(windows["article_ids"]):
        windows_by_article[article_id].append(window_index)

    article_ids = sorted(windows_by_article)
    root_rows, donor_rows = [], []
    root_articles, donor_articles = [], []

    for article_id in article_ids:
        other_articles = [a for a in article_ids if a != article_id]

        # Sample four distinct positions from this article's cached windows.
        available_rows = [
            window_index * 127 + position
            for window_index in windows_by_article[article_id]
            for position in range(127)
        ]

        for root_row in rng.sample(available_rows, 4):
            position = root_row % 127
            donor_article = rng.choice(other_articles)
            donor_window = rng.choice(windows_by_article[donor_article])
            donor_row = donor_window * 127 + position

            root_rows.append(root_row)
            donor_rows.append(donor_row)
            root_articles.append(article_id)
            donor_articles.append(donor_article)

    x_base = val_cache["inputs"][root_rows].clone()
    x_donor = val_cache["inputs"][donor_rows].clone()
    alpha = pilot_config["primary_blend_alpha"]
    x_edited = (1 - alpha) * x_base + alpha * x_donor

    with torch.no_grad():
        y_base = target_mlp(
            x_base.to(device).unsqueeze(1)
        ).squeeze(1).cpu()

        y_edited = target_mlp(
            x_edited.to(device).unsqueeze(1)
        ).squeeze(1).cpu()

    intervention_panel = {
        "x_base": x_base,
        "x_edited": x_edited,
        "true_response": y_edited - y_base,
        "root_rows": torch.tensor(root_rows),
        "donor_rows": torch.tensor(donor_rows),
        "root_articles": root_articles,
        "donor_articles": donor_articles,
        "alpha": alpha,
        "seed": 4001,
    }
    torch.save(intervention_panel, panel_path)
    print("Created and saved intervention panel.")

assert all(
    root != donor
    for root, donor in zip(
        intervention_panel["root_articles"],
        intervention_panel["donor_articles"],
    )
)

response_energy = (
    intervention_panel["true_response"].double().square().sum().item()
)
assert response_energy > 0

print("Intervention examples:", len(intervention_panel["x_base"]))
print("Root articles:", len(set(intervention_panel["root_articles"])))
print(f"Total true response energy: {response_energy:.4f}")

Created and saved intervention panel.
Intervention examples: 240
Root articles: 60
Total true response energy: 125929.0966


In [ ]:
panel_results = {}

x_base = intervention_panel["x_base"].to(device)
x_edited = intervention_panel["x_edited"].to(device)
true_response = intervention_panel["true_response"].double()

denominator = true_response.square().sum()
assert denominator > 0

with torch.no_grad():
    for name, model in pilot_models.items():
        model.eval()

        base_prediction, _ = model(x_base)
        edited_prediction, _ = model(x_edited)

        # Identity intervention: unchanged input should produce no response.
        identity_prediction, _ = model(x_base)
        identity_error = (
            identity_prediction - base_prediction
        ).abs().max().item()
        assert identity_error == 0.0

        predicted_response = (
            edited_prediction.double() - base_prediction.double()
        ).cpu()

        error = predicted_response - true_response
        squared_error_per_example = error.square().sum(dim=-1)

        normalized_error = (
            squared_error_per_example.sum() / denominator
        ).item()

        panel_results[name] = {
            "normalized_response_error": normalized_error,
            "squared_error_per_example": squared_error_per_example,
            "predicted_response": predicted_response,
            "identity_max_error": identity_error,
        }

        print(
            f"{name:15s} | Response E: {normalized_error:.4f}"
            f" | Identity error: {identity_error:.1e}"
        )

print("\nPredict-no-change baseline | Response E: 1.0000")

learned_error = panel_results["Learned"]["normalized_response_error"]

for name in ["Frozen encoder", "Frozen decoder"]:
    difference = (
        panel_results[name]["normalized_response_error"] - learned_error
    )
    print(f"{name} − Learned | D: {difference:+.4f}")

torch.save(
    {
        "step": completed_steps,
        "k": 32,
        "seed": 0,
        "panel_path": str(panel_path),
        "results": panel_results,
        "reconstruction_matched": False,
    },
    run_dir / "validation_intervention_results.pt",
)

print("\nIntervention results saved.")

Learned         | Response E: 0.8200 | Identity error: 0.0e+00
Frozen encoder  | Response E: 0.9750 | Identity error: 0.0e+00
Frozen decoder  | Response E: 0.9909 | Identity error: 0.0e+00

Predict-no-change baseline | Response E: 1.0000
Frozen encoder − Learned | D: +0.1550
Frozen decoder − Learned | D: +0.1709

Intervention results saved.


In [ ]:
# Ridge strength relative to the average training feature variance.
RIDGE_RELATIVE = 1e-6

@torch.no_grad()
def refit_decoder(source_model, batch_size=1024):
    source_model.eval()

    n_features = source_model.W_enc.shape[1]
    d_out = source_model.W_dec.shape[1]
    n_train = len(train_cache["inputs"])

    # Accumulate sufficient statistics in float64 on the GPU.
    sum_h = torch.zeros(n_features, dtype=torch.float64, device=device)
    sum_y = torch.zeros(d_out, dtype=torch.float64, device=device)
    hth = torch.zeros(
        n_features, n_features, dtype=torch.float64, device=device
    )
    hty = torch.zeros(
        n_features, d_out, dtype=torch.float64, device=device
    )

    for start in tqdm(range(0, n_train, batch_size)):
        x = train_cache["inputs"][start:start + batch_size].to(device)
        y = train_cache["outputs"][start:start + batch_size].to(
            device=device, dtype=torch.float64
        )

        _, features = source_model(x)
        h = features.double()

        sum_h += h.sum(dim=0)
        sum_y += y.sum(dim=0)
        hth.addmm_(h.T, h)
        hty.addmm_(h.T, y)

    mean_h = sum_h / n_train
    mean_y = sum_y / n_train

    # Centering lets us fit an unpenalized output bias.
    covariance = hth / n_train - torch.outer(mean_h, mean_h)
    cross_covariance = hty / n_train - torch.outer(mean_h, mean_y)
    covariance = (covariance + covariance.T) / 2

    ridge = RIDGE_RELATIVE * covariance.diagonal().mean().item()
    assert ridge > 0

    covariance.diagonal().add_(ridge)

    factor, info = torch.linalg.cholesky_ex(covariance)
    assert info.item() == 0, "Ridge system could not be factorized."

    fitted_weights = torch.cholesky_solve(cross_covariance, factor)
    fitted_bias = mean_y - mean_h @ fitted_weights

    # Preserve the original model and its training optimizer.
    refitted = copy.deepcopy(source_model)
    refitted.W_dec.copy_(fitted_weights.to(refitted.W_dec))
    refitted.b_dec_out.copy_(fitted_bias.to(refitted.b_dec_out))
    refitted.requires_grad_(False)
    refitted.eval()

    for parameter in ["W_enc", "b_enc", "b_dec"]:
        assert torch.equal(
            getattr(refitted, parameter),
            getattr(source_model, parameter),
        )

    return refitted, ridge


refitted_models = {}
refit_results = {}

for name in ["Learned", "Frozen encoder"]:
    print(f"\nRefitting: {name}")

    source = pilot_models[name]
    before = validation_metrics(source)

    refitted, ridge = refit_decoder(source)
    after = validation_metrics(refitted)

    refitted_models[name] = refitted
    refit_results[name] = {
        "before": before,
        "after": after,
        "ridge_absolute": ridge,
        "ridge_relative": RIDGE_RELATIVE,
    }

    print(
        f"Val NMSE: {before['nmse']:.6f} → {after['nmse']:.6f}\n"
        f"L0:       {before['l0']:.2f} → {after['l0']:.2f}\n"
        f"Ridge coefficient: {ridge:.3e}"
    )

    torch.save(
        {
            "state_dict": {
                key: value.detach().cpu().clone()
                for key, value in refitted.state_dict().items()
            },
            "source_variant": name,
            "source_step": completed_steps,
            "config": pilot_config,
            "k": source.k,
            "method": "Centered ridge regression; unpenalized output bias",
            "fit_split": "train",
            "metrics": refit_results[name],
        },
        run_dir / f"{name.lower().replace(' ', '_')}_ridge_refit.pt",
    )

print("\nRefitted models saved. Original model weights unchanged.")


Refitting: Learned


  0%|          | 0/127 [00:00<?, ?it/s]

Val NMSE: 0.641055 → 0.620955
L0:       32.00 → 32.00
Ridge coefficient: 1.623e-08

Refitting: Frozen encoder


  0%|          | 0/127 [00:00<?, ?it/s]

Val NMSE: 0.851585 → 0.852479
L0:       32.00 → 32.00
Ridge coefficient: 5.974e-09

Refitted models saved. Original model weights unchanged.


In [ ]:
refit_panel_results = {}

x_base = intervention_panel["x_base"].to(device)
x_edited = intervention_panel["x_edited"].to(device)
true_response = intervention_panel["true_response"].double()
denominator = true_response.square().sum()

with torch.no_grad():
    for name, model in refitted_models.items():
        model.eval()

        base_prediction, _ = model(x_base)
        edited_prediction, _ = model(x_edited)

        predicted_response = (
            edited_prediction.double() - base_prediction.double()
        ).cpu()

        error = predicted_response - true_response
        squared_error = error.square().sum(dim=-1)
        response_e = (squared_error.sum() / denominator).item()

        refit_panel_results[name] = {
            "normalized_response_error": response_e,
            "squared_error_per_example": squared_error,
            "predicted_response": predicted_response,
        }

        before = panel_results[name]["normalized_response_error"]
        print(f"{name:15s} | Response E: {before:.4f} → {response_e:.4f}")

difference = (
    refit_panel_results["Frozen encoder"]["normalized_response_error"]
    - refit_panel_results["Learned"]["normalized_response_error"]
)
print(f"\nAfter refitting | Frozen encoder − Learned: {difference:+.4f}")

torch.save(
    {
        "source_step": completed_steps,
        "k": 32,
        "seed": 0,
        "panel_path": str(panel_path),
        "results": refit_panel_results,
        "refit_metrics": refit_results,
        "reconstruction_matched": False,
    },
    run_dir / "ridge_refit_intervention_results.pt",
)

print("Results saved.")


Learned         | Response E: 0.8200 → 0.6508
Frozen encoder  | Response E: 0.9750 → 0.8504

After refitting | Frozen encoder − Learned: +0.1996
Results saved.


In [ ]:
CURRENT_K = 128
CURRENT_SEED = 0

run_dir = cache_dir.parent / "runs" / f"k{CURRENT_K}_seed{CURRENT_SEED}"
run_dir.mkdir(parents=True, exist_ok=True)
latest_path = run_dir / "latest.pt"

if latest_path.exists():
    raise RuntimeError(
        f"Checkpoint already exists at {latest_path}. Restore it instead."
    )

# Reproduce the same initial matrices; change only k.
base_model = TopKTranscoder(
    d_in=pilot_config["d_in"],
    n_features=pilot_config["n_features"],
    d_out=pilot_config["d_out"],
    k=CURRENT_K,
    seed=CURRENT_SEED,
)

pilot_models = {
    name: copy.deepcopy(base_model).to(device)
    for name in pilot_config["variants"]
}
pilot_models["Frozen encoder"].W_enc.requires_grad_(False)
pilot_models["Frozen decoder"].W_dec.requires_grad_(False)

frozen_matrices = {
    "Frozen encoder": pilot_models["Frozen encoder"].W_enc.detach().clone(),
    "Frozen decoder": pilot_models["Frozen decoder"].W_dec.detach().clone(),
}

pilot_optimizers = {
    name: torch.optim.Adam(
        [p for p in model.parameters() if p.requires_grad],
        lr=pilot_config["learning_rate"],
    )
    for name, model in pilot_models.items()
}

# Same sampling sequence as the k=32 experiment.
batch_generator = torch.Generator().manual_seed(3001)
completed_steps = 0
training_history = []

initial_metrics = {
    name: validation_metrics(model)
    for name, model in pilot_models.items()
}
best_validation = {
    name: metrics["nmse"] for name, metrics in initial_metrics.items()
}
best_steps = {name: 0 for name in pilot_models}
best_states = {
    name: {
        key: value.detach().cpu().clone()
        for key, value in model.state_dict().items()
    }
    for name, model in pilot_models.items()
}

for name, metrics in initial_metrics.items():
    print(
        f"{name:15s} | Initial NMSE: {metrics['nmse']:.4f}"
        f" | L0: {metrics['l0']:.2f}"
    )

print("\nPrepared k=64. No training performed.")

Learned         | Initial NMSE: 1.1441 | L0: 128.00
Frozen encoder  | Initial NMSE: 1.1441 | L0: 128.00
Frozen decoder  | Initial NMSE: 1.1441 | L0: 128.00

Prepared k=64. No training performed.


In [ ]:
def save_training_checkpoint():
    actual_k = next(iter(pilot_models.values())).k
    assert all(model.k == actual_k for model in pilot_models.values())
    assert actual_k == CURRENT_K

    # Derive the destination from the current experiment.
    destination = (
        cache_dir.parent / "runs" / f"k{actual_k}_seed{CURRENT_SEED}"
    )
    destination.mkdir(parents=True, exist_ok=True)

    payload = {
        "config": copy.deepcopy(pilot_config),
        "k": actual_k,
        "seed": CURRENT_SEED,
        "completed_steps": completed_steps,
        "models": {
            name: {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            for name, model in pilot_models.items()
        },
        "optimizers": {
            name: optimizer.state_dict()
            for name, optimizer in pilot_optimizers.items()
        },
        "requires_grad": {
            name: {
                key: parameter.requires_grad
                for key, parameter in model.named_parameters()
            }
            for name, model in pilot_models.items()
        },
        "batch_generator_state": batch_generator.get_state(),
        "frozen_matrices": {
            name: tensor.detach().cpu().clone()
            for name, tensor in frozen_matrices.items()
        },
        "history": training_history,
        "best_validation": best_validation,
        "best_states": best_states,
        "best_steps": best_steps,
        "sampling": "uniform token sampling with replacement",
    }

    temporary_path = destination / "latest.tmp.pt"
    torch.save(payload, temporary_path)
    temporary_path.replace(destination / "latest.pt")

In [ ]:
save_training_checkpoint()
train_cache = activation_cache["train"]
TARGET_STEP = pilot_config["training_steps"]

for step in tqdm(range(completed_steps + 1, TARGET_STEP + 1)):
    indices = torch.randint(
        len(train_cache["inputs"]),
        (pilot_config["batch_size"],),
        generator=batch_generator,
    )
    x = train_cache["inputs"][indices].to(device)
    y = train_cache["outputs"][indices].to(device)

    for name, model in pilot_models.items():
        model.train()
        optimizer = pilot_optimizers[name]
        optimizer.zero_grad(set_to_none=True)

        prediction, _ = model(x)
        loss = F.mse_loss(prediction, y)
        assert torch.isfinite(loss), f"Nonfinite loss: {name}"

        loss.backward()
        optimizer.step()

    completed_steps = step

    if step % 100 == 0:
        assert torch.equal(
            pilot_models["Frozen encoder"].W_enc,
            frozen_matrices["Frozen encoder"],
        )
        assert torch.equal(
            pilot_models["Frozen decoder"].W_dec,
            frozen_matrices["Frozen decoder"],
        )

        for name, model in pilot_models.items():
            metrics = validation_metrics(model)
            training_history.append({
                "step": step, "variant": name, **metrics
            })

            if metrics["nmse"] < best_validation[name]:
                best_validation[name] = metrics["nmse"]
                best_steps[name] = step
                best_states[name] = {
                    key: value.detach().cpu().clone()
                    for key, value in model.state_dict().items()
                }

            if step % 500 == 0:
                print(
                    f"Step {step} | {name:15s}"
                    f" | Val NMSE: {metrics['nmse']:.4f}"
                    f" | L0: {metrics['l0']:.2f}"
                    f" | Inactive: {metrics['inactive_fraction']:.1%}"
                )

        save_training_checkpoint()

print("\nTraining complete; checkpoint saved:", latest_path)

  0%|          | 0/2000 [00:00<?, ?it/s]

Step 500 | Learned         | Val NMSE: 0.7359 | L0: 128.00 | Inactive: 0.3%
Step 500 | Frozen encoder  | Val NMSE: 0.8451 | L0: 128.00 | Inactive: 0.1%
Step 500 | Frozen decoder  | Val NMSE: 0.9270 | L0: 128.00 | Inactive: 0.6%
Step 1000 | Learned         | Val NMSE: 0.6584 | L0: 128.00 | Inactive: 0.1%
Step 1000 | Frozen encoder  | Val NMSE: 0.7994 | L0: 128.00 | Inactive: 0.1%
Step 1000 | Frozen decoder  | Val NMSE: 0.9013 | L0: 128.00 | Inactive: 1.0%
Step 1500 | Learned         | Val NMSE: 0.6162 | L0: 128.00 | Inactive: 0.1%
Step 1500 | Frozen encoder  | Val NMSE: 0.7825 | L0: 128.00 | Inactive: 0.1%
Step 1500 | Frozen decoder  | Val NMSE: 0.8870 | L0: 128.00 | Inactive: 0.8%
Step 2000 | Learned         | Val NMSE: 0.5884 | L0: 128.00 | Inactive: 0.1%
Step 2000 | Frozen encoder  | Val NMSE: 0.7757 | L0: 128.00 | Inactive: 0.1%
Step 2000 | Frozen decoder  | Val NMSE: 0.8773 | L0: 128.00 | Inactive: 0.7%

Training complete; checkpoint saved: /content/drive/MyDrive/transcoder_pilot/g

In [ ]:
assert completed_steps == 2000
assert all(model.k == CURRENT_K for model in pilot_models.values())

@torch.no_grad()
def evaluate_response(model):
    model.eval()

    x_base = intervention_panel["x_base"].to(device)
    x_edited = intervention_panel["x_edited"].to(device)
    true_response = intervention_panel["true_response"].double()

    base_prediction, _ = model(x_base)
    edited_prediction, _ = model(x_edited)

    predicted_response = (
        edited_prediction.double() - base_prediction.double()
    ).cpu()

    squared_error = (
        predicted_response - true_response
    ).square().sum(dim=-1)

    return {
        "normalized_response_error": (
            squared_error.sum() / true_response.square().sum()
        ).item(),
        "squared_error_per_example": squared_error,
        "predicted_response": predicted_response,
    }


# Original trained models.
panel_results = {}
for name, model in pilot_models.items():
    panel_results[name] = evaluate_response(model)
    print(
        f"{name:15s} | Original response E: "
        f"{panel_results[name]['normalized_response_error']:.4f}"
    )

# Separate decoder-refitted models.
refitted_models = {}
refit_results = {}
refit_panel_results = {}

for name in ["Learned", "Frozen encoder"]:
    print(f"\nRefitting: {name}")

    before = validation_metrics(pilot_models[name])
    refitted, ridge = refit_decoder(pilot_models[name])
    after = validation_metrics(refitted)
    response = evaluate_response(refitted)

    refitted_models[name] = refitted
    refit_results[name] = {
        "before": before,
        "after": after,
        "ridge_absolute": ridge,
        "ridge_relative": RIDGE_RELATIVE,
    }
    refit_panel_results[name] = response

    print(f"Val NMSE: {before['nmse']:.6f} → {after['nmse']:.6f}")
    print(
        "Response E: "
        f"{panel_results[name]['normalized_response_error']:.4f}"
        f" → {response['normalized_response_error']:.4f}"
    )

    torch.save(
        {
            "state_dict": {
                key: value.detach().cpu().clone()
                for key, value in refitted.state_dict().items()
            },
            "source_variant": name,
            "source_step": completed_steps,
            "config": pilot_config,
            "k": refitted.k,
            "seed": CURRENT_SEED,
            "method": "Centered ridge regression; unpenalized output bias",
            "fit_split": "train",
            "metrics": refit_results[name],
        },
        run_dir / f"{name.lower().replace(' ', '_')}_ridge_refit.pt",
    )

torch.save(
    {
        "k": CURRENT_K,
        "seed": CURRENT_SEED,
        "step": completed_steps,
        "panel_path": str(panel_path),
        "original_results": panel_results,
        "refitted_results": refit_panel_results,
        "refit_metrics": refit_results,
        "reconstruction_matched": False,
    },
    run_dir / "intervention_comparison.pt",
)

print("\nResults and refitted models saved:", run_dir)


Learned         | Original response E: 0.7474
Frozen encoder  | Original response E: 0.9286
Frozen decoder  | Original response E: 0.9696

Refitting: Learned


  0%|          | 0/127 [00:00<?, ?it/s]

Val NMSE: 0.588394 → 0.568271
Response E: 0.7474 → 0.5654

Refitting: Frozen encoder


  0%|          | 0/127 [00:00<?, ?it/s]

Val NMSE: 0.775750 → 0.770386
Response E: 0.9286 → 0.7495

Results and refitted models saved: /content/drive/MyDrive/transcoder_pilot/gpt2_block5/runs/k128_seed0


| \(k\) | Learned NMSE | Frozen encoder NMSE | Learned response \(E\) | Frozen encoder response \(E\) | Advantage \(D\) |
|---:|---:|---:|---:|---:|---:|
| 32 | 0.6210 | 0.8525 | 0.6508 | 0.8504 | +0.1996 |
| 64 | 0.5891 | 0.8064 | 0.5960 | 0.8002 | +0.2042 |
| 128 | 0.5683 | 0.7704 | 0.5654 | 0.7495 | +0.1841 |

In [ ]:
runs_root = cache_dir.parent / "runs"

for k in [32, 64, 128]:
    folder = runs_root / f"k{k}_seed0"
    required = [
        "latest.pt",
        "learned_ridge_refit.pt",
        "frozen_encoder_ridge_refit.pt",
    ]
    results_file = (
        "ridge_refit_intervention_results.pt"
        if k == 32 else "intervention_comparison.pt"
    )

    for filename in required + [results_file]:
        path = folder / filename
        assert path.exists() and path.stat().st_size > 0, f"Missing: {path}"

    print(f"k={k}: checkpoints and results present.")

print("\nReady to disconnect. Also save the notebook with Ctrl+S.")


k=32: checkpoints and results present.
k=64: checkpoints and results present.
k=128: checkpoints and results present.

Ready to disconnect. Also save the notebook with Ctrl+S.
